# Exploration and sanity checks

Visual checks that the data is what we think it is, before trusting any accuracy number.
Run top to bottom. Every cell answers a specific question that, if answered wrong,
invalidates everything downstream.

**Milestone 1 of the project plan:** load + epoch one subject, plot the raw signal,
confirm the event markers make sense.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from src.preprocess import (
    load_physionet_subject, preprocess_raw, epoch_physionet, epochs_to_xy,
)
from src.config import SUB_BANDS, CANONICAL_SFREQ

%matplotlib inline

## 1. Load one subject

Runs 4/8/12 are imagined left- vs right-fist movement. Check that the annotations
really are T0/T1/T2 rather than assuming it — the dataset's convention is easy to
misremember, and getting T1/T2 backwards inverts every label.

In [ ]:
raw = load_physionet_subject(1)
print(raw)
print("sampling rate:", raw.info["sfreq"], "Hz")
print("channels:", len(raw.ch_names))

import collections
print("annotations:", collections.Counter(raw.annotations.description))
print("T0 = rest, T1 = imagine LEFT fist, T2 = imagine RIGHT fist")

## 2. Look at the raw signal

Before filtering. You should see slow drift and, if you scroll, blinks on the frontal
channels. If a channel is flat or railed, find out now.

In [ ]:
raw.plot(duration=10, n_channels=12, scalings="auto", title="Raw EEG (unfiltered)");

## 3. Power spectrum — is this even EEG?

A real EEG spectrum falls off roughly as 1/f with a bump around 10 Hz (alpha).
A flat spectrum means you are looking at noise; a spectrum dominated by a 60 Hz
spike means powerline contamination.

**This is the first check to run on new headband data too.**

In [ ]:
raw.compute_psd(fmax=80).plot();

## 4. Preprocess and epoch

8-30 Hz bandpass (mu + beta), common average reference, resample to 128 Hz,
cut 0.5-3.5 s after each cue.

In [ ]:
clean = preprocess_raw(raw)
epochs = epoch_physionet(clean, tmin=-2.0, tmax=4.0)
X, y = epochs_to_xy(epoch_physionet(clean))

print("epochs:", X.shape, "(trials, channels, samples)")
print("class balance:", dict(zip(["left", "right"], np.bincount(y))))
print("\nIf these are badly imbalanced, artifact rejection is hitting one class")
print("harder than the other, which inflates accuracy.")

## 5. The money plot: does C3 vs C4 mu power actually differ?

This is the whole hypothesis in one figure. Imagining the **left** hand should
suppress mu power at **C4** (right hemisphere); imagining the right should suppress
it at C3.

If the bars do not cross, this subject is not producing a usable ERD — and no
classifier will rescue that.

In [ ]:
from scipy.signal import welch

ch_idx = {name: epochs.ch_names.index(name) for name in ["C3", "C4"]}
active = epochs.copy().crop(0.5, 3.5)

fig, ax = plt.subplots(figsize=(6, 4))
width, offsets = 0.35, {"left": -0.18, "right": 0.18}

for cls, off in offsets.items():
    data = active[cls].get_data(copy=True)
    freqs, psd = welch(data, fs=active.info["sfreq"], nperseg=64, axis=-1)
    mu = (freqs >= 8) & (freqs <= 13)
    powers = [psd[:, ch_idx[c], mu].mean(axis=-1) for c in ["C3", "C4"]]
    ax.bar(
        np.arange(2) + off,
        [p.mean() * 1e12 for p in powers],
        yerr=[p.std() / np.sqrt(len(p)) * 1e12 for p in powers],
        width=width, capsize=4, label=f"imagine {cls}",
    )

ax.set_xticks([0, 1], ["C3 (left hemi)", "C4 (right hemi)"])
ax.set_ylabel("mu power (uV^2/Hz)")
ax.set_title("Contralateral mu suppression")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## 6. ERD topomap

The spatial version of the same question. Blue = power decrease = cortex engaged.
Expect blue over the hemisphere **opposite** the imagined hand.

Frontal or symmetric blobs mean you are looking at eye movement or general
attention, not motor imagery.

In [ ]:
from src.evaluate import plot_erd_topomap
from IPython.display import Image

path = plot_erd_topomap(epochs)
Image(str(path))

## 7. What CSP learned

CSP patterns should look like lateralised blobs near C3/C4. If the top pattern
loads on a frontal electrode, suspect eye artefact.

In [ ]:
from src.features import make_csp

csp = make_csp(n_components=4)
csp.fit(X, y)
csp.plot_patterns(epochs.info, ch_type="eeg", units="a.u.", size=1.5);

## 8. Simulate a headband

Restrict the 64-channel data to the electrodes a real device has, and see what
accuracy survives. This is how to price-check a purchase before making it.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

from src.models import build_model
from src.train import resolve_channels

cv = StratifiedKFold(5, shuffle=True, random_state=42)

for device in [None, "crown", "unicorn", "ganglion", "muse2"]:
    try:
        picks = resolve_channels(device, raw.ch_names)
        Xd, yd = epochs_to_xy(epoch_physionet(preprocess_raw(raw, pick_channels=picks)))
        score = cross_val_score(build_model("csp_lda"), Xd, yd, cv=cv).mean()
        label = device or "sensorimotor strip"
        print(f"{label:22s} {len(picks):2d} ch  CSP+LDA = {score:.3f}")
    except Exception as exc:
        print(f"{device:22s} -> {exc}")

Note what happens with `muse2`: its electrodes barely intersect the dataset montage
at all, and none of the overlap is over motor cortex. That is the argument against
buying one for this project, made quantitatively.

See [`docs/HARDWARE.md`](../docs/HARDWARE.md) for the full buying guide.